In [12]:
import json
import pandas as pd

def jprint(obj):
    print(json.dumps(obj, indent=2))


In [13]:
with open('/home/baris/.cache/huggingface/hub/datasets--bdsaglam--predictions-ragent-Qwen2.5-1.5B-Instruct-musique-grpo/snapshots/eab98df035c11b20a8c84ec361debe955ac27d9b/predictions-ragent-Qwen2.5-1.5B-Instruct-musique-grpo.jsonl') as f:
    records = [json.loads(line) for line in f]
print(len(records))

300


In [14]:
df = pd.DataFrame(records)
df.head()

,answer,n_hops,prompt,docs,answers,supporting_titles,trajectory
0,Vito Corleone,2,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...","[Vito Corleone, Vito Andolini, Vito Andolini C...","[The Good Shepherd (film), The Godfather Part II]",[{'content': 'Answer the question based on the...
1,Rohana Wijeweera,2,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...",[Rohana Wijeweera],"[Rohana Wijeweera, Dimuthu Bandara Abayakoon]",[{'content': 'Answer the question based on the...
2,406,2,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...",[406],"[Rössen culture, Galicia (Spain)]",[{'content': 'Answer the question based on the...
3,Jamie Murray,2,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...",[Jamie Murray],"[2011 Valencia Open 500 – Doubles, 2016 Wimble...",[{'content': 'Answer the question based on the...
4,modern-day Italians,2,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...",[modern-day Italians],"[Jews, Ashkenazi Jews]",[{'content': 'Answer the question based on the...


In [15]:
from verifiers.rubrics.utils import get_last_answer

df['n_hops'] = df['supporting_titles'].apply(len)
df['predicted_answer'] = df['trajectory'].apply(get_last_answer)

In [16]:
from verifiers.metrics.musique import exact_match, f1

df['exact_match'] = df.apply(lambda row: exact_match(row['predicted_answer'], row['answers']), axis=1)
df['f1'] = df.apply(lambda row: f1(row['predicted_answer'], row['answers']), axis=1)

df.groupby('n_hops')[['exact_match', 'f1']].agg(['count', 'mean', 'std'])

exact_match                    f1                    
             count  mean       std count      mean       std
n_hops                                                      
2              100  0.18  0.386123   100  0.245723  0.395783
3              100  0.05  0.219043   100  0.098041  0.255815
4              100  0.03  0.171447   100  0.071833  0.227977

In [17]:
import textwrap
from ipywidgets import widgets
from IPython.display import display
from ipywidgets import HBox
from tabulate import tabulate

def fixedwidth(text):
    """Wrap text to fixed width"""
    return "\n".join(textwrap.wrap(str(text), width=120, replace_whitespace=False))

def format_messages(messages):
    """Format a list of messages as a chat"""
    return "\n".join([f"{m['role']}: {m['content']}" for m in messages])

def format_row(row):
    """Format a single row for display as a table"""
    n_prefix_messages = len(row['prompt'])
    data = [
        ["F1", f"{row['f1']:.2f}"],
        ["Answers", row['answers']],
        ["Predicted Answer", row['predicted_answer']],
        ["Prompt", fixedwidth(format_messages(row['prompt'][-1:]))], 
        ["Trajectory", fixedwidth(format_messages(row['trajectory'][n_prefix_messages:]))],
    ]
    return tabulate(data, tablefmt='grid')

def present_row(row):
    """Display a formatted row as table"""
    print(format_row(row))

def create_browse_app(df):
    """Create an interactive widget to browse through the data"""
    def browse_data(i=0):
        row = df.iloc[i]
        present_row(row)

    index = widgets.IntText(value=0, description='Index:')
    left_button = widgets.Button(description='Previous')
    right_button = widgets.Button(description='Next')

    def on_left_button_clicked(b):
        if index.value > 0:
            index.value -= 1

    def on_right_button_clicked(b):
        if index.value < len(df) - 1:
            index.value += 1

    left_button.on_click(on_left_button_clicked)
    right_button.on_click(on_right_button_clicked)

    ui = HBox([left_button, index, right_button])
    out = widgets.interactive_output(browse_data, {'i': index})

    display(ui, out)

In [18]:
mask = df['n_hops'] == 2
sdf = df[mask]
create_browse_app(sdf)

Output()